# Fine-tune จับทรงกล่องจากไดไลน์ v2.1 (Colab GPU)

**EfficientNet-B3** + ข้อมูล proxy (cap/ทรง) + **eval บน adjudicated holdout** (per-case audit)

**⚠️ อ่านก่อน (ตาม GPT review):**
- holdout 9 ใบ = **expert 4 + claude-review 5** (ไม่ใช่ expert ทั้งหมด) → รายงานแยก source
- holdout ครอบเพียง **class 1,2,9,12** → เป็น **per-case audit เท่านั้น ไม่ใช่ 12-way accuracy** ห้ามสรุป "B3 ชนะ B0" จากชุดนี้
- train = proxy label (noisy ~33% ผิดบน custom) | leakage กันด้วย connected-components family (prep แล้ว)
- **multi-seed (3 seeds)** + **best-checkpoint (เลือกจาก proxy-val ไม่ใช่ holdout)** → วัด B3-vs-B0 ให้สม่ำเสมอ
- baseline: CLIP-kNN 43%, fine-tune B0 = 51%/tuck 57%

**ขั้นตอน:** Runtime→GPU(T4) → Run all → อัป `dieline_train.zip`


In [ ]:
!pip -q install timm scikit-learn


### 1. อัปโหลด dieline_train.zip


In [ ]:
from google.colab import files
import zipfile, os
up = files.upload()
zipfile.ZipFile(list(up.keys())[0]).extractall('data')
print('train:', len(os.listdir('data/img')), '| gold:', len(os.listdir('data/gold')))


### 2. โหลด labels + group-split (train/val ภายใน)


In [ ]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
df = pd.read_csv('data/labels.csv'); df['y']=df['label']-1
gold = pd.read_csv('data/gold_labels.csv'); gold['y']=gold['label']-1
tr,va = next(GroupShuffleSplit(1,test_size=0.15,random_state=42).split(df,groups=df['group_id']))
train_df,val_df = df.iloc[tr].reset_index(drop=True), df.iloc[va].reset_index(drop=True)
assert not (set(train_df.group_id)&set(val_df.group_id)), 'val leak'
print('train',len(train_df),'| val',len(val_df),'| families',df.group_id.nunique())
print('train ต่อทรง:',train_df.label.value_counts().sort_index().to_dict())
print('holdout source:',gold.source.value_counts().to_dict(),'| ต่อทรง:',gold.label.value_counts().sort_index().to_dict())


### 3. Dataset


In [ ]:
import torch, timm, numpy as np, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
SZ=320
aug=T.Compose([T.Resize((SZ,SZ)),T.RandomRotation(4),T.ColorJitter(0.1,0.1),T.ToTensor(),T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
plain=T.Compose([T.Resize((SZ,SZ)),T.ToTensor(),T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])])
class DS(Dataset):
    def __init__(s,d,folder,tf): s.d=d; s.f=folder; s.tf=tf
    def __len__(s): return len(s.d)
    def __getitem__(s,i):
        r=s.d.iloc[i]; im=Image.open(f'data/{s.f}/{r.file}').convert('RGB')
        return s.tf(im), int(r.y)
tl=DataLoader(DS(train_df,'img',aug),batch_size=24,shuffle=True,num_workers=2)
vl=DataLoader(DS(val_df,'img',plain),batch_size=48,num_workers=2)
gl=DataLoader(DS(gold,'gold',plain),batch_size=16)


### 4. เทรน (EfficientNet-B3, class-weighted, multi-seed + best-checkpoint)
seed ครบ (torch/numpy/random/cudnn) · เก็บ best_state จาก **proxy-val** (ไม่แตะ holdout) · รัน 3 seeds รายงาน mean/range


In [ ]:
import random, statistics
dev='cuda' if torch.cuda.is_available() else 'cpu'; print(dev)
cnt=train_df.y.value_counts().sort_index()
w=torch.tensor([(len(train_df)/(12*cnt.get(i,1)))**0.5 for i in range(12)],dtype=torch.float).clamp(max=4).to(dev)  # sqrt+cap

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False

def val_acc(m):
    m.eval(); c=n=0
    with torch.no_grad():
        for x,y in vl: c+=(m(x.to(dev)).argmax(1).cpu()==y).sum().item(); n+=len(y)
    return 100*c/n

def train_one(seed, epochs=20):
    seed_all(seed)
    m=timm.create_model('efficientnet_b3',pretrained=True,num_classes=12).to(dev)
    crit=nn.CrossEntropyLoss(weight=w,label_smoothing=0.05)
    opt=torch.optim.AdamW(m.parameters(),lr=3e-4,weight_decay=1e-4)
    sch=torch.optim.lr_scheduler.OneCycleLR(opt,max_lr=3e-4,total_steps=epochs*len(tl),pct_start=0.15)
    best_val=-1; best_state=None
    for ep in range(epochs):
        m.train(); tot=0
        for x,y in tl:
            x,y=x.to(dev),y.to(dev); opt.zero_grad(); loss=crit(m(x),y); loss.backward(); opt.step(); sch.step(); tot+=loss.item()
        va=val_acc(m)
        if va>best_val: best_val=va; best_state={k:v.detach().cpu().clone() for k,v in m.state_dict().items()}
        print('  seed%d ep%2d loss%.3f val %.1f%% (best %.1f%%)'%(seed,ep+1,tot/len(tl),va,best_val))
    m.load_state_dict(best_state)
    return m,best_val

SEEDS=[42,1,7]
runs=[]; best=(-1,None,None)
for s in SEEDS:
    m,bv=train_one(s); runs.append((s,bv))
    if bv>best[0]: best=(bv,m,s)
print('\n=== proxy-val best/seed (group-safe) — B0 51% เป็น historical reference (คนละ split, ไม่ head-to-head) ===')
for s,bv in runs: print('  seed %d: %.1f%%'%(s,bv))
vals=[bv for _,bv in runs]
print('  mean %.1f%% | range %.1f–%.1f%%'%(statistics.mean(vals),min(vals),max(vals)))
model=best[1]; print('>> เลือก seed %d (val %.1f%%) → ประเมิน holdout (เลือกจาก proxy-val เท่านั้น)'%(best[2],best[0]))


### 5. ประเมินบน proxy val (เทียบ baseline)


In [ ]:
from sklearn.metrics import classification_report
model.eval(); P=[]; Y=[]
with torch.no_grad():
    for x,y in vl: P+=(model(x.to(dev)).argmax(1).cpu()+1).tolist(); Y+=(y+1).tolist()
P,Y=np.array(P),np.array(Y)
print('=== บน proxy val (noisy label — วัด "ตรง proxy" ไม่ใช่ "ถูกจริง") ===')
print('12-way: %.0f%%'%(100*(P==Y).mean()))
tk=np.isin(Y,[1,2,4,11]); print('tuck-family: %.0f%% (n=%d)'%(100*(P[tk]==Y[tk]).mean(),tk.sum()))
print('custom-vs-std: %.0f%%'%(100*((P==12)==(Y==12)).mean()))
print('  (baseline B0 51% / CLIP-kNN 43% = historical, คนละ split — ยังไม่ head-to-head จนกว่าจะ rerun บน family split เดียวกัน)')


### 6. 🎯 ประเมินบน adjudicated holdout — **per-case AUDIT** (แยก expert vs claude)
⚠️ ไม่ใช่ "12-way accuracy" (ครอบแค่ 4 class) — ดูว่าโมเดล **แก้** custom ที่ proxy ผิด จริงไหม


In [ ]:
model.eval(); GP=[]
with torch.no_grad():
    for x,y in gl: GP+=(model(x.to(dev)).argmax(1).cpu()+1).tolist()
g=gold.copy(); g['pred']=GP; g['ok']=g['pred']==g['label']
print('=== adjudicated holdout (%d ใบ) — per-case AUDIT (ไม่ใช่ 12-way accuracy) ==='%len(g))
for src in ['expert','claude']:
    d=g[g.source==src]
    if len(d): print('  %-7s n=%d: %d/%d ถูก'%(src,len(d),int(d.ok.sum()),len(d)))
print('  combined n=%d: %d/%d = %.0f%%  (Claude/CV บนชุดนี้ = 44%%)'%(len(g),int(g.ok.sum()),len(g),100*g.ok.mean()))
print('  ต่อใบ (label[src] -> ทาย):')
for _,r in g.iterrows(): print('    %2d[%s] -> %2d %s'%(r.label,r.source[:3],r.pred,'✓' if r.ok else '✗'))
cust=g[g.label==12]
print('  >> custom(12) ที่ proxy มัก under-label: %d/%d โมเดลตีถูก'%(int(cust.ok.sum()),len(cust)))
print('  ⚠️ ครอบเพียง class %s → ห้ามสรุป "B3 12-way ชนะ B0" จากชุดนี้'%sorted(g.label.unique()))


### 7. เซฟโมเดล


In [ ]:
torch.save(model.state_dict(),'dieline_effb3.pt')
from google.colab import files; files.download('dieline_effb3.pt')
